# Лабораторная работа №16: Визуальные токенизаторы и autoregressive image generation

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Обучить compact VQ-VAE.
- Измерить reconstruction и codebook utilization.
- Обучить autoregressive prior и сравнить sampling.

## Теоретическая часть

VQ-VAE кодирует изображение в grid latent vectors, заменяет каждый ближайшим codebook vector и декодирует. Индексы становятся visual tokens для autoregressive prior. Малый codebook ухудшает reconstruction, большой может иметь dead codes.

![VQ-VAE](https://raw.githubusercontent.com/CompVis/taming-transformers/master/assets/teaser.png)


VQ-VAE заменяет пространственные признаки ближайшими векторами кодбука, straight-through переносит градиент через дискретизацию. Conditional autoregressive prior предсказывает очередной visual token по уже известным индексам и метке класса.

Для индекса $k^*=\arg\min_k\|z_e-e_k\|_2^2$ straight-through задаёт $z_q=z_e+\operatorname{sg}(e_{k^*}-z_e)$. Perplexity кодов равна $\exp(-\sum_k p_k\log p_k)$.

### Экспериментальный протокол

MNIST train 20 000/test 2 000; 28×28→7×7 visual tokens. Два токенизатора K=16/64; обученный conditional Transformer; 1000 шагов каждого в полном режиме. Smoke 2 шага — только техническая проверка.

### Материалы

- [VQ-VAE](https://arxiv.org/abs/1711.00937)
- [Taming Transformers](https://github.com/CompVis/taming-transformers)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True   # одинаковые числа при повторном прогоне
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = ARTIFACTS / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')  # UTF-8 обязательно (Windows-локаль cp1251)
print({'device': DEVICE, 'fast_dev_run': FAST_DEV_RUN, 'artifacts': str(ARTIFACTS),
       'torch': torch.__version__, 'numpy': np.__version__})

{'device': 'cuda', 'fast_dev_run': False, 'artifacts': 'artifacts', 'torch': '2.14.1+cu126', 'numpy': '2.4.6'}


In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print('SSL: проверка сертификатов через системное хранилище ОС (truststore)')
except ImportError:
    print('SSL: truststore не установлен — стандартная проверка сертификатов')

SSL: проверка сертификатов через системное хранилище ОС (truststore)


In [3]:
# Проверка зависимостей лабы 16: torch, torchvision (MNIST), numpy, matplotlib.
# diffusers для этой лабы НЕ нужен. MNIST при отсутствии кэша скачается автоматически (~65 МБ,
# ossci-зеркало) в DATA_ROOT/mnist; в лабе 14 он уже скачивался - скорее всего, кэш на месте.
import importlib.util
_need = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
missing = [k for k in _need if importlib.util.find_spec(k) is None]
if missing:
    raise ImportError('Установите недостающее и перезапустите ячейку:  pip install ' + ' '.join(_need[k] for k in missing))
import torch, torchvision, numpy, matplotlib
_info = {'torch': torch.__version__, 'torchvision': torchvision.__version__,
         'numpy': numpy.__version__, 'matplotlib': matplotlib.__version__,
         'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    _info['gpu'] = f'{_p.name} | cc {_p.major}.{_p.minor}'
else:
    _info['gpu_note'] = 'CUDA недоступна - допустимо: модели лабы крошечные, полный прогон идёт за минуты даже на CPU'
print('Версии:', _info)

Версии: {'torch': '2.14.1+cu126', 'torchvision': '0.29.1+cu126', 'numpy': '2.4.6', 'matplotlib': '3.11.2', 'device': 'cuda', 'gpu': 'NVIDIA GeForce GTX 1050 Ti | cc 6.1'}


### Архитектура дискретного токенизатора

*Вес: 1 балл.*

**Постановка.** Кодбук должен обучаться на реальных изображениях; random inputs не измеряют качество реконструкции. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите VQ-VAE, ближайший код, straight-through и loss с корректными масштабами.

**Результат.** Проверяемое состояние «Архитектура дискретного токенизатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите VQ-VAE, ближайший код, straight-through и loss с корректными масштабами» и его результат доступен следующему этапу.

In [4]:
# ЗАДАНИЕ 1 - архитектура дискретного токенизатора VQ-VAE: энкодер -> латентная сетка 7x7,
# кодбук, квантование ближайшим кодом, straight-through и loss с корректными масштабами.
import torch.nn as nn
import torch.nn.functional as F

IMG_CH, IMG_SIZE = 1, 28
LAT_WH = 7                       # 28 -> 14 -> 7 (две stride-2 свёртки): 7*7 = 49 visual tokens на изображение
LAT_DIM = 64                     # размерность латентных векторов z_e и векторов кодбука
COMMIT_BETA = 0.25               # вес commitment-члена (van den Oord et al., 2017)

class VQEncoder(nn.Module):
    def __init__(self, z_dim=LAT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(IMG_CH, 32, 4, stride=2, padding=1), nn.ReLU(inplace=True),   # 28 -> 14
            nn.Conv2d(32, 64, 4, stride=2, padding=1), nn.ReLU(inplace=True),       # 14 -> 7
            nn.Conv2d(64, z_dim, 3, stride=1, padding=1))                           # 7x7 x z_dim

    def forward(self, x):
        return self.net(x)

class VQDecoder(nn.Module):
    def __init__(self, z_dim=LAT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(z_dim, 64, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),                            # 7 -> 14
            nn.Conv2d(64, 32, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='nearest'),                            # 14 -> 28
            nn.Conv2d(32, IMG_CH, 3, stride=1, padding=1), nn.Sigmoid())            # выход в [0,1] - как x

    def forward(self, z_q):
        return self.net(z_q)

class VQVAE(nn.Module):
    # Дискретный токенизатор: квантование ближайшим кодом кодбука + straight-through.
    def __init__(self, n_codes, z_dim=LAT_DIM):
        super().__init__()
        self.n_codes = n_codes
        self.encoder = VQEncoder(z_dim)
        self.decoder = VQDecoder(z_dim)
        self.codebook = nn.Embedding(n_codes, z_dim)
        nn.init.uniform_(self.codebook.weight, -1.0 / n_codes, 1.0 / n_codes)  # инициализация как в статье VQ-VAE

    def encode(self, x):
        return self.encoder(x)                                   # z_e: [B, z_dim, 7, 7]

    def quantize(self, z_e):
        # Ближайший код в L2: argmin_k ||z_e - e_k||^2 = argmin (||z_e||^2 - 2<z_e,e_k> + ||e_k||^2)
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)            # [B*49, D]
        d2 = (flat.pow(2).sum(1, keepdim=True) - 2 * flat @ self.codebook.weight.T
              + self.codebook.weight.pow(2).sum(1)[None, :])     # [B*49, K]
        idx = d2.argmin(dim=1)                                   # ближайший код для каждой позиции
        z_q = self.codebook(idx).view(B, H, W, D).permute(0, 3, 1, 2)
        loss_code = F.mse_loss(self.codebook(idx), flat.detach())          # коды -> к z_e
        loss_commit = COMMIT_BETA * F.mse_loss(flat, self.codebook(idx).detach())  # z_e -> к кодам (beta=0.25)
        z_q_st = z_e + (z_q - z_e).detach()                       # straight-through: градиент декодера течёт в энкодер как при тождественном отображении
        return z_q_st, idx.view(B, H, W), loss_code, loss_commit

    def decode(self, z_q):
        return self.decoder(z_q)

    def forward(self, x):
        z_e = self.encode(x)
        z_q_st, idx, loss_code, loss_commit = self.quantize(z_e)
        return self.decode(z_q_st), idx, loss_code, loss_commit

# Smoke на детерминированном шуме - только проверка связности; обучение позже строго на реальном MNIST (Задание 2).
_vq_smoke = VQVAE(n_codes=16)
_x = torch.rand(4, IMG_CH, IMG_SIZE, IMG_SIZE, generator=torch.Generator().manual_seed(SEED))
_rec, _idx, _lc, _lm = _vq_smoke(_x)
assert _rec.shape == _x.shape and _idx.shape == (4, LAT_WH, LAT_WH)
assert (_idx >= 0).all() and (_idx < 16).all()
_vq_smoke.zero_grad(); (_rec.mean() + _lc + _lm).backward()
assert _vq_smoke.encoder.net[0].weight.grad is not None      # straight-through доводит градиент до энкодера
assert _vq_smoke.codebook.weight.grad is not None            # кодбук учитывается через loss_code/loss_commit
_n_enc = sum(p.numel() for p in _vq_smoke.encoder.parameters())
_n_dec = sum(p.numel() for p in _vq_smoke.decoder.parameters())
print('ЗАДАНИЕ 1 OK: VQ-VAE | энкодер 28x28 -> 7x7 x', LAT_DIM, '| параметры энкодера', _n_enc, '| декодера', _n_dec)
print('loss = recon MSE + ||sg(z_e) - e||^2 +', COMMIT_BETA, '* ||z_e - sg(e)||^2; квантование ближайшим кодом + straight-through')

ЗАДАНИЕ 1 OK: VQ-VAE | энкодер 28x28 -> 7x7 x 64 | параметры энкодера 70304 | декодера 55681
loss = recon MSE + ||sg(z_e) - e||^2 + 0.25 * ||z_e - sg(e)||^2; квантование ближайшим кодом + straight-through


### Обучение и диагностика двух кодбуков

*Вес: 1 балл.*

**Постановка.** Кодбук должен обучаться на реальных изображениях; random inputs не измеряют качество реконструкции. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите K=16/64 при равном бюджете; измерьте test MSE, used/dead/perplexity и сохраните checkpoint.

**Результат.** Проверяемое состояние «Обучение и диагностика двух кодбуков» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите K=16/64 при равном бюджете; измерьте test MSE, used/dead/perplexity и сохраните checkpoint» и его результат доступен следующему этапу.

In [5]:
# ЗАДАНИЕ 2 - MNIST train 20000 / test 2000 (сбалансированные классы) и обучение двух кодбуков
# K=16/64 при РАВНОМ бюджете: одинаковая инициализация энкодера/декодера, общий поток батчей, одинаковые шаги;
# диагностика test MSE / used / dead / perplexity; чекпоинты с resume.
import torchvision

MNIST_DIR = DATA_ROOT / 'mnist'
_mnist_train_raw = torchvision.datasets.MNIST(str(MNIST_DIR), train=True, download=True)
_mnist_test_raw = torchvision.datasets.MNIST(str(MNIST_DIR), train=False, download=True)

def balanced_subset(data, targets, per_class, seed):
    # детерминированная сбалансированная выборка: per_class изображений на каждый класс
    rs = np.random.RandomState(seed)
    data = np.asarray(data); targets = np.asarray(targets)
    keep = np.concatenate([rs.choice(np.where(targets == c)[0], size=per_class, replace=False) for c in range(10)])
    x = torch.from_numpy(data[keep]).float().unsqueeze(1) / 255.0   # [N,1,28,28] в [0,1]
    return x, torch.from_numpy(targets[keep]).long()

TRAIN_PER_CLASS = 20 if FAST_DEV_RUN else 2000    # полный режим: 20000 train
TEST_PER_CLASS = 20 if FAST_DEV_RUN else 200      # полный режим: 2000 test
TRAIN_X, TRAIN_Y = balanced_subset(_mnist_train_raw.data, _mnist_train_raw.targets, TRAIN_PER_CLASS, SEED + 1)
TEST_X, TEST_Y = balanced_subset(_mnist_test_raw.data, _mnist_test_raw.targets, TEST_PER_CLASS, SEED + 2)
print('данные: train', tuple(TRAIN_X.shape), '/ test', tuple(TEST_X.shape),
      '| диапазон [0,1]:', round(float(TRAIN_X.min()), 1), '-', round(float(TRAIN_X.max()), 1))

VQ_STEPS = 2 if FAST_DEV_RUN else 1000
VQ_BATCH = 256
VQ_LR = 3e-4
VQ_CODES = (16, 64)

def make_batches(seed, n, steps, batch):
    # общий поток батчей для обоих K: одна и та же последовательность индексов = равный бюджет
    g = torch.Generator().manual_seed(seed)
    return [torch.randint(0, n, (batch,), generator=g) for _ in range(steps)]

def train_vqvae(n_codes, steps, batch_seed, init_seed):
    torch.manual_seed(init_seed)   # одинаковая инициализация энкодера/декодера у обоих K (различие - только размер кодбука)
    model = VQVAE(n_codes=n_codes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=VQ_LR)
    batches = make_batches(batch_seed, TRAIN_X.size(0), steps, VQ_BATCH)
    ckpt_path = ARTIFACTS / f'lab16_ckpt_vq{n_codes}.pt'
    start_step, history = 0, []
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer'])
        start_step, history = ck['step'], ck['loss_history']
        print(f'  K={n_codes}: resume с шага {start_step}')
    model.train()
    t0 = time.time()
    for step in range(start_step, steps):
        x = TRAIN_X[batches[step]].to(DEVICE)
        x_rec, idx, loss_code, loss_commit = model(x)
        loss = F.mse_loss(x_rec, x) + loss_code + loss_commit
        opt.zero_grad(); loss.backward(); opt.step()
        history.append(float(loss.item()))
    train_time = time.time() - t0
    torch.save({'model': model.state_dict(), 'optimizer': opt.state_dict(), 'step': steps,
                'loss_history': history, 'n_codes': n_codes, 'steps': steps, 'lr': VQ_LR}, ckpt_path)
    return model, history, train_time

@torch.no_grad()
def vq_diagnostics(model, x_all):
    # test MSE (шкала [0,1]), used/dead коды и perplexity распределения назначений по кодам
    model.eval()
    mse_sum, mse_n = 0.0, 0
    counts = torch.zeros(model.n_codes, dtype=torch.long)
    for i in range(0, x_all.size(0), 512):
        x = x_all[i:i + 512].to(DEVICE)
        x_rec, idx, _, _ = model(x)
        mse_sum += F.mse_loss(x_rec, x, reduction='sum').item(); mse_n += x.numel()
        counts += torch.bincount(idx.reshape(-1).cpu(), minlength=model.n_codes)
    p = counts.double(); p = p / p.sum(); p = p[p > 0]
    return {'test_mse': mse_sum / mse_n, 'used': int((counts > 0).sum()),
            'dead': int(model.n_codes - (counts > 0).sum()), 'perplexity': float(torch.exp(-(p * p.log()).sum()))}

VQ_MODELS, VQ_RESULTS = {}, {}
for _K in VQ_CODES:
    _m, _h, _t = train_vqvae(_K, VQ_STEPS, batch_seed=SEED + 20, init_seed=SEED + 100)
    _d = vq_diagnostics(_m, TEST_X)
    VQ_MODELS[_K] = _m
    VQ_RESULTS[_K] = {'steps': VQ_STEPS, 'batch': VQ_BATCH, 'lr': VQ_LR, 'loss_first': round(_h[0], 4),
                      'loss_last': round(_h[-1], 4), 'train_time_s': round(_t, 1), **_d}
    print(f'K={_K}: loss {_h[0]:.4f} -> {_h[-1]:.4f} за {VQ_STEPS} шагов | test MSE {_d["test_mse"]:.5f} | '
          f'used {_d["used"]}/{_K} | dead {_d["dead"]} | perplexity {_d["perplexity"]:.2f} | {round(_t, 1)} c')

import matplotlib.pyplot as plt
_shown = TEST_X[:8]
_f, _ax = plt.subplots(3, 8, figsize=(12, 4.6), constrained_layout=True)
_rows = [('оригинал', _shown), ('K=16', None), ('K=64', None)]
for _K, _r in ((16, 1), (64, 2)):
    VQ_MODELS[_K].eval()
    with torch.no_grad():
        _rows[_r] = (f'реконструкция K={_K}', VQ_MODELS[_K](_shown.to(DEVICE))[0].cpu())
for _r, (_title, _imgs) in enumerate(_rows):
    for _c in range(8):
        _ax[_r, _c].imshow(_imgs[_c, 0], cmap='gray', vmin=0, vmax=1); _ax[_r, _c].axis('off')
    _ax[_r, 0].set_ylabel(_title, fontsize=9)
_f.savefig(ARTIFACTS / 'lab16_vq_recon.png', dpi=120)
plt.close(_f)
save_json('lab16_vq_report.json', {'seed_init': SEED + 100, 'seed_batches': SEED + 20, 'per_k': VQ_RESULTS})
print('PNG:', ARTIFACTS / 'lab16_vq_recon.png', '| отчёт: lab16_vq_report.json')

данные: train (20000, 1, 28, 28) / test (2000, 1, 28, 28) | диапазон [0,1]: 0.0 - 1.0
  K=16: resume с шага 2
K=16: loss 0.2477 -> 0.0709 за 1000 шагов | test MSE 0.01254 | used 16/16 | dead 0 | perplexity 9.42 | 26.3 c
  K=64: resume с шага 2
K=64: loss 0.2472 -> 0.0427 за 1000 шагов | test MSE 0.01053 | used 34/64 | dead 30 | perplexity 13.48 | 26.3 c
PNG: artifacts\lab16_vq_recon.png | отчёт: lab16_vq_report.json


### Индексы обученного токенизатора

*Вес: 1 балл.*

**Постановка.** Prior обучается исключительно на индексах обученного токенизатора, без доступа к будущим позициям. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Закодируйте train/test только обученной VQ-VAE K=64.

**Результат.** Проверяемое состояние «Индексы обученного токенизатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Закодируйте train/test только обученной VQ-VAE K=64» и его результат доступен следующему этапу.

In [6]:
# ЗАДАНИЕ 3 - индексы обученного токенизатора: train/test кодируются ТОЛЬКО обученной VQ-VAE K=64.
N_CODES_PRIOR = 64
assert N_CODES_PRIOR in VQ_MODELS, 'K=64 должен быть обучен в Задании 2'
_vq64 = VQ_MODELS[N_CODES_PRIOR].eval()

@torch.no_grad()
def encode_tokens(model, x_all):
    out = []
    for i in range(0, x_all.size(0), 512):
        _, idx, _, _ = model.quantize(model.encode(x_all[i:i + 512].to(DEVICE)))
        out.append(idx.reshape(idx.size(0), -1).cpu())
    return torch.cat(out)                                    # [N, 49] int64

TRAIN_TOKENS = encode_tokens(_vq64, TRAIN_X)
TEST_TOKENS = encode_tokens(_vq64, TEST_X)
assert TRAIN_TOKENS.shape == (TRAIN_X.size(0), LAT_WH * LAT_WH) and TEST_TOKENS.shape == (TEST_X.size(0), LAT_WH * LAT_WH)
assert TRAIN_TOKENS.min() >= 0 and TRAIN_TOKENS.max() < N_CODES_PRIOR
assert torch.equal(encode_tokens(_vq64, TRAIN_X[:32]), TRAIN_TOKENS[:32]), 'повторное кодирование должно быть детерминированным'
_counts = torch.bincount(TRAIN_TOKENS.reshape(-1), minlength=N_CODES_PRIOR).double()
_p = _counts / _counts.sum(); _p = _p[_p > 0]
TRAIN_TOKEN_PERPLEXITY = float(torch.exp(-(_p * _p.log()).sum()))
TRAIN_TOKEN_USED = int((_counts > 0).sum())
print('ЗАДАНИЕ 3 OK: train', tuple(TRAIN_TOKENS.shape), '| test', tuple(TEST_TOKENS.shape),
      '| used кодов на train:', TRAIN_TOKEN_USED, 'из', N_CODES_PRIOR,
      '| perplexity токенов:', round(TRAIN_TOKEN_PERPLEXITY, 2))
np.savez_compressed(ARTIFACTS / 'lab16_tokens_train64.npz', indices=TRAIN_TOKENS.numpy(), labels=TRAIN_Y.numpy())
np.savez_compressed(ARTIFACTS / 'lab16_tokens_test64.npz', indices=TEST_TOKENS.numpy(), labels=TEST_Y.numpy())
print('NPZ:', ARTIFACTS / 'lab16_tokens_train64.npz', '|', ARTIFACTS / 'lab16_tokens_test64.npz')

ЗАДАНИЕ 3 OK: train (20000, 49) | test (2000, 49) | used кодов на train: 41 из 64 | perplexity токенов: 13.54
NPZ: artifacts\lab16_tokens_train64.npz | artifacts\lab16_tokens_test64.npz


### Причинная маска prior

*Вес: 1 балл.*

**Постановка.** Prior обучается исключительно на индексах обученного токенизатора, без доступа к будущим позициям. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Создайте Transformer с BOS и маской, исключающей просмотр будущих токенов.

**Результат.** Проверяемое состояние «Причинная маска prior» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Создайте Transformer с BOS и маской, исключающей просмотр будущих токенов» и его результат доступен следующему этапу.

In [7]:
# ЗАДАНИЕ 4 - причинный Transformer-prior: BOS, learned позиции, условие на класс, маска без просмотра будущего.
SEQ_LEN = LAT_WH * LAT_WH          # 49 токенов на изображение
BOS_ID = N_CODES_PRIOR             # служебный токен начала последовательности
PRIOR_VOCAB = N_CODES_PRIOR + 1    # на входе 64 кода + BOS; на выходе предсказываем только коды (0..63)
D_MODEL, N_HEADS, N_LAYERS, D_FF = 128, 4, 2, 256

class PriorTransformer(nn.Module):
    def __init__(self, vocab, n_positions, n_codes, d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, d_ff=D_FF, n_classes=10):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d_model)
        self.pos_emb = nn.Embedding(n_positions, d_model)          # n_positions = 50: BOS + 49 токенов
        self.cls_emb = nn.Embedding(n_classes, d_model)            # условие: метка класса цифры
        _layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=d_ff,
                                            dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(_layer, n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_codes)                    # логиты только по кодам 0..63
        mask = torch.triu(torch.ones(n_positions, n_positions, dtype=torch.bool), diagonal=1)
        self.register_buffer('attn_mask', mask)                    # True = позиция ЗАПРЕЩЕНА (будущее)

    def forward(self, idx, labels):
        h = self.tok_emb(idx) + self.pos_emb.weight[: idx.size(1)][None] + self.cls_emb(labels)[:, None, :]
        h = self.blocks(h, mask=self.attn_mask[: idx.size(1), : idx.size(1)])
        return self.head(self.norm(h))                             # [B, L, 64]

PRIOR = PriorTransformer(PRIOR_VOCAB, SEQ_LEN + 1, N_CODES_PRIOR).to(DEVICE)
# Проверка причинности: изменение БУДУЩЕГО токена не влияет на логиты ПРЕДЫДУЩИХ позиций.
PRIOR.eval()
_a = torch.randint(0, PRIOR_VOCAB, (2, SEQ_LEN + 1), generator=torch.Generator().manual_seed(SEED + 3)).to(DEVICE)  # генерация на CPU (детерминизм) -> на устройство PRIOR
_a[:, 0] = BOS_ID
_b = _a.clone(); _b[1, 35] = (_b[1, 35] + 1) % PRIOR_VOCAB     # ломаем токен в позиции 35 последовательности 1
_lab = torch.tensor([0, 1], device=DEVICE)
with torch.no_grad():
    _o1, _o2 = PRIOR(_a, _lab), PRIOR(_b, _lab)
assert torch.allclose(_o1[:, :35], _o2[:, :35], atol=1e-4), 'утечка из будущего: логиты позиций 0..34 изменились!'
assert not torch.allclose(_o1[:, 35:], _o2[:, 35:], atol=1e-4), 'проверка фиктивна: изменение ни на что не повлияло'
print('ЗАДАНИЕ 4 OK: причинная маска проверена (позиции 0..34 не видят токен 35 и будущее), BOS =', BOS_ID)
print('параметры prior:', sum(p.numel() for p in PRIOR.parameters()),
      '| d_model', D_MODEL, '| слоёв', N_LAYERS, '| голов', N_HEADS, '| ff', D_FF)

ЗАДАНИЕ 4 OK: причинная маска проверена (позиции 0..34 не видят токен 35 и будущее), BOS = 64
параметры prior: 289472 | d_model 128 | слоёв 2 | голов 4 | ff 256


C:\Users\eric1\AppData\Local\Temp\ipykernel_21788\3859262871.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(_layer, n_layers)


### Обучение autoregressive prior

*Вес: 1 балл.*

**Постановка.** Prior обучается исключительно на индексах обученного токенизатора, без доступа к будущим позициям. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Оптимизируйте CE сдвинутой последовательности с checkpoint.

**Результат.** Проверяемое состояние «Обучение autoregressive prior» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Оптимизируйте CE сдвинутой последовательности с checkpoint» и его результат доступен следующему этапу.

In [8]:
# ЗАДАНИЕ 5 - обучение autoregressive prior: CE по сдвинутой последовательности, чекпоинт с resume.
PRIOR_STEPS = 2 if FAST_DEV_RUN else 1000
PRIOR_BATCH = 256
PRIOR_LR = 1e-3

_prior_opt = torch.optim.Adam(PRIOR.parameters(), lr=PRIOR_LR)
_prior_ckpt = ARTIFACTS / 'lab16_ckpt_prior.pt'
_pg = torch.Generator().manual_seed(SEED + 30)
_prior_batches = [torch.randint(0, TRAIN_TOKENS.size(0), (PRIOR_BATCH,), generator=_pg) for _ in range(PRIOR_STEPS)]
_prior_history, _start = [], 0
if _prior_ckpt.exists():
    _ck = torch.load(_prior_ckpt, map_location=DEVICE, weights_only=False)
    PRIOR.load_state_dict(_ck['model']); _prior_opt.load_state_dict(_ck['optimizer'])
    _start, _prior_history = _ck['step'], _ck['loss_history']
    print('prior: resume с шага', _start)
PRIOR.train()
_t0 = time.time()
for _step in range(_start, PRIOR_STEPS):
    _idx = _prior_batches[_step]
    _tok, _lab = TRAIN_TOKENS[_idx].to(DEVICE), TRAIN_Y[_idx].to(DEVICE)
    _inp = torch.cat([torch.full((_tok.size(0), 1), BOS_ID, dtype=torch.long, device=_tok.device), _tok], dim=1)  # [B, 50]
    _logits = PRIOR(_inp, _lab)[:, :-1]                        # позиция i предсказывает токен i+1
    _loss = F.cross_entropy(_logits.reshape(-1, N_CODES_PRIOR), _tok.reshape(-1))
    _prior_opt.zero_grad(); _loss.backward(); _prior_opt.step()
    _prior_history.append(float(_loss.item()))
PRIOR_TIME = time.time() - _t0
torch.save({'model': PRIOR.state_dict(), 'optimizer': _prior_opt.state_dict(), 'step': PRIOR_STEPS,
            'loss_history': _prior_history, 'steps': PRIOR_STEPS, 'lr': PRIOR_LR}, _prior_ckpt)
save_json('lab16_prior_training_log.json', {'steps': PRIOR_STEPS, 'batch': PRIOR_BATCH, 'lr': PRIOR_LR,
           'loss_first': _prior_history[0], 'loss_last': _prior_history[-1],
           'ce_perplexity': float(np.exp(_prior_history[-1])),
           'train_time_s': round(PRIOR_TIME, 1), 'history_every_50': _prior_history[::50]})
print(f'ЗАДАНИЕ 5 OK: CE {_prior_history[0]:.4f} -> {_prior_history[-1]:.4f} '
      f'(perplexity {np.exp(_prior_history[-1]):.2f} кодов) за {PRIOR_STEPS} шагов, {PRIOR_TIME:.1f} c')

prior: resume с шага 2
ЗАДАНИЕ 5 OK: CE 4.3785 -> 0.9150 (perplexity 2.50 кодов) за 1000 шагов, 48.2 c


### Cross-entropy на test токенах

*Вес: 1 балл.*

**Постановка.** Prior обучается исключительно на индексах обученного токенизатора, без доступа к будущим позициям. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Оцените CE независимых test токенов без обновления весов.

**Результат.** Проверяемое состояние «Cross-entropy на test токенах» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Оцените CE независимых test токенов без обновления весов» и его результат доступен следующему этапу.

In [9]:
# ЗАДАНИЕ 6 - cross-entropy на независимых test-токенах БЕЗ обновления весов + базовые ориентиры.
PRIOR.eval()

@torch.no_grad()
def prior_ce(tokens, labels):
    tot, n = 0.0, 0
    for i in range(0, tokens.size(0), 512):
        tok, lab = tokens[i:i + 512].to(DEVICE), labels[i:i + 512].to(DEVICE)
        inp = torch.cat([torch.full((tok.size(0), 1), BOS_ID, dtype=torch.long, device=tok.device), tok], dim=1)
        logits = PRIOR(inp, lab)[:, :-1]
        tot += F.cross_entropy(logits.reshape(-1, N_CODES_PRIOR), tok.reshape(-1), reduction='sum').item()
        n += tok.numel()
    return tot / n

TEST_CE = prior_ce(TEST_TOKENS, TEST_Y)
TRAIN_CE_HOLDIN = prior_ce(TRAIN_TOKENS[:2048], TRAIN_Y[:2048])   # контроль переобучения на hold-in срезе train
_uniform_ce = float(np.log(N_CODES_PRIOR))
_uc = torch.bincount(TRAIN_TOKENS.reshape(-1), minlength=N_CODES_PRIOR).double()
_up = _uc / _uc.sum(); _up = _up[_up > 0]
_unigram_ce = float(-(_up * _up.log()).sum())
if not FAST_DEV_RUN:
    assert TEST_CE < _unigram_ce, 'prior не должен быть хуже униграммного ориентира на test'
save_json('lab16_prior_test_ce.json', {'test_ce': TEST_CE, 'test_perplexity': float(np.exp(TEST_CE)),
           'train_ce_hold_in': TRAIN_CE_HOLDIN, 'unigram_ce': _unigram_ce, 'uniform_ce': _uniform_ce})
print(f'ЗАДАНИЕ 6 OK: test CE {TEST_CE:.4f} (perplexity {np.exp(TEST_CE):.2f}) | train(hold-in) {TRAIN_CE_HOLDIN:.4f} '
      f'| униграмма {_unigram_ce:.4f} | равномерное ln(64) {_uniform_ce:.4f}')

ЗАДАНИЕ 6 OK: test CE 0.9416 (perplexity 2.56) | train(hold-in) 0.8839 | униграмма 2.6057 | равномерное ln(64) 4.1589


### Архитектура проверяющего классификатора

*Вес: 1 балл.*

**Постановка.** Сравниваются декодированные изображения, а не распределения случайных токенов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите отдельную сеть для class agreement.

**Результат.** Проверяемое состояние «Архитектура проверяющего классификатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите отдельную сеть для class agreement» и его результат доступен следующему этапу.

In [10]:
# ЗАДАНИЕ 7 - независимый проверяющий классификатор для class agreement (DigitCNN, как в лабе 14).
class DigitCNN(nn.Module):
    # Вход: [B,1,28,28] в [0,1] - тот же диапазон, что у реальных MNIST и декодированных сэмплов.
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),    # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2))   # 14 -> 7
        self.classifier = nn.Linear(32 * 7 * 7, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

CLF = DigitCNN().to(DEVICE)
_z = CLF(torch.rand(2, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 4)).to(DEVICE))
assert _z.shape == (2, 10)
CLF_PARAMS = sum(p.numel() for p in CLF.parameters())
print('ЗАДАНИЕ 7 OK: DigitCNN', CLF_PARAMS, 'параметров (независим и от VQ-VAE, и от prior)')

ЗАДАНИЕ 7 OK: DigitCNN 20490 параметров (независим и от VQ-VAE, и от prior)


### Обучение и реальный test классификатора

*Вес: 1 балл.*

**Постановка.** Сравниваются декодированные изображения, а не распределения случайных токенов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите на train и проверьте точность на независимом real test.

**Результат.** Проверяемое состояние «Обучение и реальный test классификатора» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите на train и проверьте точность на независимом real test» и его результат доступен следующему этапу.

In [11]:
# ЗАДАНИЕ 8 - обучение проверяющего классификатора на реальном train и точность на независимом real test.
CLS_STEPS = 2 if FAST_DEV_RUN else 1000
CLS_BATCH = 256
_clf_opt = torch.optim.Adam(CLF.parameters(), lr=1e-3)
_cg = torch.Generator().manual_seed(SEED + 40)
_clf_batches = [torch.randint(0, TRAIN_X.size(0), (CLS_BATCH,), generator=_cg) for _ in range(CLS_STEPS)]
_clf_ckpt = ARTIFACTS / 'lab16_ckpt_classifier.pt'
_start = 0
if _clf_ckpt.exists():
    _ck = torch.load(_clf_ckpt, map_location=DEVICE, weights_only=False)
    CLF.load_state_dict(_ck['model']); _clf_opt.load_state_dict(_ck['optimizer']); _start = _ck['step']
    print('классификатор: resume с шага', _start)
CLF.train()
_t0 = time.time()
for _step in range(_start, CLS_STEPS):
    _idx = _clf_batches[_step]
    _x, _y = TRAIN_X[_idx].to(DEVICE), TRAIN_Y[_idx].to(DEVICE)
    _loss = F.cross_entropy(CLF(_x), _y)
    _clf_opt.zero_grad(); _loss.backward(); _clf_opt.step()
CLS_TIME = time.time() - _t0

@torch.no_grad()
def clf_eval(x_all, y_all):
    CLF.eval()
    correct = torch.zeros(10, dtype=torch.long); total = torch.zeros(10, dtype=torch.long)
    conf_sum, n = 0.0, 0
    for i in range(0, x_all.size(0), 512):
        x, y = x_all[i:i + 512].to(DEVICE), y_all[i:i + 512].to(DEVICE)
        probs = F.softmax(CLF(x), dim=1)
        pred = probs.argmax(1)
        conf_sum += float(probs.max(1).values.sum().item()); n += y.numel()
        for c in range(10):
            m = y == c
            total[c] += int(m.sum()); correct[c] += int((pred[m] == c).sum())
    return float(correct.sum().item() / total.sum().item()), correct.tolist(), total.tolist(), conf_sum / n

CLF_ACC, CLF_CORRECT, CLF_TOTAL, CLF_CONF = clf_eval(TEST_X, TEST_Y)
torch.save({'model': CLF.state_dict(), 'optimizer': _clf_opt.state_dict(), 'step': CLS_STEPS}, _clf_ckpt)
save_json('lab16_classifier_report.json', {'steps': CLS_STEPS, 'batch': CLS_BATCH, 'params': CLF_PARAMS,
           'test_accuracy': CLF_ACC, 'mean_confidence': CLF_CONF,
           'per_class_correct': CLF_CORRECT, 'per_class_total': CLF_TOTAL, 'train_time_s': round(CLS_TIME, 1)})
print(f'ЗАДАНИЕ 8 OK: точность на независимом real test {CLF_ACC:.4f} '
      f'({int(sum(CLF_CORRECT))}/{int(sum(CLF_TOTAL))}), средняя уверенность {CLF_CONF:.4f}, {CLS_TIME:.1f} c')

классификатор: resume с шага 2
ЗАДАНИЕ 8 OK: точность на независимом real test 0.9825 (1965/2000), средняя уверенность 0.9812, 8.7 c


### Фильтрация и семплирование токенов

*Вес: 1 балл.*

**Постановка.** Сравниваются декодированные изображения, а не распределения случайных токенов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Реализуйте temperature, top-k и top-p, генерируя индексы авторегрессионно.

**Результат.** Проверяемое состояние «Фильтрация и семплирование токенов» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Реализуйте temperature, top-k и top-p, генерируя индексы авторегрессионно» и его результат доступен следующему этапу.

In [12]:
# ЗАДАНИЕ 9 - фильтрация распределения (temperature, top-k, top-p) и авторегрессивное семплирование индексов.
def filter_logits(logits, temperature=1.0, top_k=0, top_p=1.0):
    # порядок: температура -> top-k -> top-p; отфильтрованные позиции получают -inf
    out = logits / max(float(temperature), 1e-6)
    if top_k and int(top_k) > 0:
        kth = torch.topk(out, min(int(top_k), out.size(-1)), dim=-1).values[..., -1, None]
        out = out.masked_fill(out < kth, float('-inf'))          # при равных логитах все ties остаются
    if 0.0 < float(top_p) < 1.0:
        srt, srt_idx = torch.sort(out, dim=-1, descending=True)
        cum = F.softmax(srt, dim=-1).cumsum(-1)
        drop = (cum - F.softmax(srt, dim=-1)) >= float(top_p)    # оставляем минимальный префикс с массой >= top_p
        srt = srt.masked_fill(drop, float('-inf'))
        out = torch.full_like(out, float('-inf')).scatter(-1, srt_idx, srt)
    return out

def make_gen(seed):
    # детерминированный генератор на устройстве семплирования (CUDA/CPU)
    return torch.Generator(device=DEVICE).manual_seed(seed) if DEVICE == 'cuda' else torch.Generator().manual_seed(seed)

def sample_tokens(model, labels, temperature=1.0, top_k=0, top_p=1.0, generator=None):
    # авторегрессивная генерация SEQ_LEN индексов: каждый следующий токен - только из уже сгенерированных (причинная маска)
    model.eval()
    labels = labels.to(DEVICE)                               # метки могут прийти CPU-тензором (model на DEVICE)
    idx = torch.full((labels.size(0), 1), BOS_ID, dtype=torch.long, device=labels.device)
    with torch.no_grad():
        for _ in range(SEQ_LEN):
            logits = model(idx, labels)[:, -1]                   # предсказание следующего токена
            filtered = filter_logits(logits, temperature, top_k, top_p)
            probs = F.softmax(filtered, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
    return idx[:, 1:]                                            # [B, 49] без BOS

# юнит-проверки фильтров
_l = torch.randn(4, N_CODES_PRIOR, generator=torch.Generator().manual_seed(SEED + 5))
_f = filter_logits(_l, temperature=0.8)
assert torch.allclose(F.softmax(_f, -1), F.softmax(_l / 0.8, -1), atol=1e-6)      # температура - монотонное перемасштабирование
_fk = filter_logits(_l, top_k=5)
assert int(torch.isfinite(_fk).sum()) == 4 * 5                                     # ровно top_k позиций оставлено
_fp = filter_logits(_l, top_p=0.5)
_mass = F.softmax(_fp, -1).sum(-1)
assert (_mass >= 0.5 - 1e-5).all() and (_mass <= 1.0 + 1e-5).all() and int(torch.isfinite(_fp[0]).sum()) >= 1
_fc = filter_logits(_l, temperature=0.9, top_k=10, top_p=0.9)
assert torch.allclose(F.softmax(_fc, -1).sum(-1), torch.ones(4), atol=1e-5)        # комбинация нормирована
_toks = sample_tokens(PRIOR, torch.arange(10), temperature=1.0, generator=make_gen(SEED + 6))
assert _toks.shape == (10, SEQ_LEN) and _toks.min() >= 0 and _toks.max() < N_CODES_PRIOR
print('ЗАДАНИЕ 9 OK: temperature/top-k/top-p проверены юнит-ассертами; авторегрессивная генерация', tuple(_toks.shape))

ЗАДАНИЕ 9 OK: temperature/top-k/top-p проверены юнит-ассертами; авторегрессивная генерация (10, 49)


### Декодированная генерация и проверка

*Вес: 1 балл.*

**Постановка.** Сравниваются декодированные изображения, а не распределения случайных токенов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сравните четыре схемы на реальных декодированных изображениях; сохраните измерения.

**Результат.** Проверяемое состояние «Декодированная генерация и проверка» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сравните четыре схемы на реальных декодированных изображениях; сохраните измерения» и его результат доступен следующему этапу.

In [13]:
# ЗАДАНИЕ 10 - сравнение ЧЕТЫРЁХ схем сэмплирования на реальных декодированных изображениях:
# class agreement у независимого классификатора, уверенность, разнообразие и близость к test; PNG-сетки, отчёт.
SCHEMES = {'base': dict(temperature=1.0, top_k=0, top_p=1.0),
           'lowT': dict(temperature=0.75, top_k=0, top_p=1.0),
           'topk': dict(temperature=1.0, top_k=20, top_p=1.0),
           'topp': dict(temperature=1.0, top_k=0, top_p=0.90)}   # схемы зафиксированы ДО просмотра результатов
N_PER_CLASS = 1 if FAST_DEV_RUN else 8
GEN_LABELS = torch.arange(10).repeat_interleave(N_PER_CLASS)     # 10 классов x N_PER_CLASS

@torch.no_grad()
def decode_tokens(model, tokens):
    # tokens [B, 49] -> кодбук -> декодер -> изображения [B,1,28,28] в [0,1]
    B = tokens.size(0)
    emb = model.codebook(tokens.to(DEVICE)).view(B, LAT_WH, LAT_WH, LAT_DIM).permute(0, 3, 1, 2)
    return model.decoder(emb).cpu()

@torch.no_grad()
def nearest_test_l2(imgs):
    # средняя L2-дистанция до ближайшего реального test-изображения (проксиметр близости к manifold данных)
    flat = imgs.reshape(imgs.size(0), -1).to(DEVICE)
    tflat = TEST_X.reshape(TEST_X.size(0), -1).to(DEVICE)
    out = []
    for i in range(0, flat.size(0), 64):
        out.append(torch.cdist(flat[i:i + 64], tflat).min(1).values.cpu())
    return float(torch.cat(out).mean())

@torch.no_grad()
def real_test_nearest_l2(n_ref=200):
    # ориентир для интерпретации: средняя L2 реальных test-изображений до ближайшего ДРУГОГО реального test
    n_ref = min(n_ref, TEST_X.size(0) // 2)                  # guard маленького test (FAST): ref и pool непусты; в полном режиме no-op
    ref = TEST_X[:n_ref].reshape(n_ref, -1).to(DEVICE)
    pool = TEST_X[n_ref:].reshape(TEST_X.size(0) - n_ref, -1).to(DEVICE)
    out = []
    for i in range(0, ref.size(0), 64):
        out.append(torch.cdist(ref[i:i + 64], pool).min(1).values.cpu())
    return float(torch.cat(out).mean())

REAL_NEAR = real_test_nearest_l2()
print(f'ориентир: средняя L2 real-test до ближайшего другого real-test {REAL_NEAR:.3f} '
      '(nearest_test_l2 сэмплов сопоставим с ориентиром => сэмплы не дальше от manifold, чем реальные цифры друг от друга)')

GEN_REPORT = {}
for _name, _cfg in SCHEMES.items():
    _toks = sample_tokens(PRIOR, GEN_LABELS, generator=make_gen(SEED + 50), **_cfg)
    _imgs = decode_tokens(_vq64, _toks)
    _probs = F.softmax(CLF(_imgs.to(DEVICE)), dim=1)
    _pred = _probs.argmax(1).cpu()
    _agree = float((_pred == GEN_LABELS).double().mean())
    _conf = float(_probs.max(1).values.mean().item())
    _divs = []
    for c in range(10):
        _ic = _imgs[GEN_LABELS == c].reshape(-1, IMG_SIZE * IMG_SIZE)
        if _ic.size(0) >= 2:
            _d = torch.cdist(_ic, _ic)
            _divs.append(float(_d[~torch.eye(_ic.size(0), dtype=bool)].mean()))
    _div = float(np.mean(_divs)) if _divs else None
    _near = nearest_test_l2(_imgs)
    GEN_REPORT[_name] = {'cfg': _cfg, 'agreement': _agree, 'confidence': _conf,
                         'diversity_l2': _div, 'nearest_test_l2': _near}
    print(f'{_name:>5}: T={_cfg["temperature"]}, k={_cfg["top_k"]}, p={_cfg["top_p"]} | agreement {_agree:.3f} | '
          f'уверенность {_conf:.3f} | разнообразие L2 {round(_div, 3) if _div is not None else "-"} | до test {_near:.3f}')
    _f, _ax = plt.subplots(10, N_PER_CLASS, figsize=(1.3 * N_PER_CLASS, 13.5), squeeze=False, constrained_layout=True)
    for c in range(10):
        for j in range(N_PER_CLASS):
            _ax[c, j].imshow(_imgs[c * N_PER_CLASS + j, 0], cmap='gray', vmin=0, vmax=1)
            _ax[c, j].axis('off')
    _f.savefig(ARTIFACTS / f'lab16_gen_{_name}.png', dpi=110)
    plt.close(_f)

save_json('lab16_generation_report.json', {'n_per_class': N_PER_CLASS, 'seed_sampling': SEED + 50,
           'real_test_nearest_l2': REAL_NEAR, 'schemes': GEN_REPORT})
print('PNG:', ', '.join(f'lab16_gen_{_n}.png' for _n in SCHEMES), '| отчёт: lab16_generation_report.json')

ориентир: средняя L2 real-test до ближайшего другого real-test 8.059 (nearest_test_l2 сэмплов сопоставим с ориентиром => сэмплы не дальше от manifold, чем реальные цифры друг от друга)
 base: T=1.0, k=0, p=1.0 | agreement 0.838 | уверенность 0.927 | разнообразие L2 8.021 | до test 5.381
 lowT: T=0.75, k=0, p=1.0 | agreement 0.938 | уверенность 0.953 | разнообразие L2 7.388 | до test 4.877
 topk: T=1.0, k=20, p=1.0 | agreement 0.850 | уверенность 0.926 | разнообразие L2 8.045 | до test 5.385
 topp: T=1.0, k=0, p=0.9 | agreement 0.925 | уверенность 0.958 | разнообразие L2 7.595 | до test 4.964
PNG: lab16_gen_base.png, lab16_gen_lowT.png, lab16_gen_topk.png, lab16_gen_topp.png | отчёт: lab16_generation_report.json


### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

In [14]:
# БОНУС - гипотеза (сформулирована ДО просмотра отложенных результатов): периодическая реактивация
# dead-кодов (сброс кодов без назначений за последние 100 шагов на случайные z_e текущего батча каждые
# 100 шагов) при том же seed, данных и вычислительном бюджете снизит число dead-кодов у K=64 и не ухудшит MSE.
BONUS_STEPS = VQ_STEPS
REINIT_EVERY, USAGE_WIN = 100, 100
torch.manual_seed(SEED + 100)                                    # та же инициализация, что у base K=64 в Задании 2
_vq_b = VQVAE(n_codes=64).to(DEVICE)
_opt_b = torch.optim.Adam(_vq_b.parameters(), lr=VQ_LR)
_b_batches = make_batches(SEED + 20, TRAIN_X.size(0), BONUS_STEPS, VQ_BATCH)   # общий поток батчей с base
_gr = torch.Generator().manual_seed(SEED + 60)                   # отдельный генератор сбросов (не трогает поток батчей)
_usage = torch.zeros(64, dtype=torch.long)
_b_hist, _b_reinits = [], 0
_vq_b.train()
_t0 = time.time()
for _step in range(BONUS_STEPS):
    _x = TRAIN_X[_b_batches[_step]].to(DEVICE)
    _z_e = _vq_b.encode(_x)
    _zq, _idx, _lc, _lm = _vq_b.quantize(_z_e)
    _loss = F.mse_loss(_vq_b.decode(_zq), _x) + _lc + _lm
    _opt_b.zero_grad(); _loss.backward(); _opt_b.step()
    _b_hist.append(float(_loss.item()))
    _usage += torch.bincount(_idx.reshape(-1).cpu(), minlength=64)
    if (_step + 1) % REINIT_EVERY == 0 and (_step + 1) < BONUS_STEPS:
        _dead = (_usage == 0).nonzero().flatten()
        if _dead.numel() > 0:
            with torch.no_grad():
                _flat = _z_e.permute(0, 2, 3, 1).reshape(-1, LAT_DIM)
                _pick = torch.randint(0, _flat.size(0), (_dead.numel(),), generator=_gr)
                _vq_b.codebook.weight[_dead.to(DEVICE)] = _flat[_pick.to(DEVICE)].detach()
            _b_reinits += 1
        _usage.zero_()                                           # окно использования - последние USAGE_WIN шагов
BONUS_TIME = time.time() - _t0
_b_diag = vq_diagnostics(_vq_b, TEST_X)
_base = VQ_RESULTS[64]
_confirmed = bool(_b_diag['dead'] < _base['dead'] and _b_diag['test_mse'] < _base['test_mse'])
bonus = {'hypothesis': 'периодическая реактивация dead-кодов при том же seed/данных/бюджете снижает dead и test MSE у K=64',
         'reinit_every': REINIT_EVERY, 'usage_window': USAGE_WIN, 'reinit_events': _b_reinits,
         'base': {'test_mse': _base['test_mse'], 'used': _base['used'], 'dead': _base['dead'], 'perplexity': _base['perplexity']},
         'reactivated': {'test_mse': _b_diag['test_mse'], 'used': _b_diag['used'], 'dead': _b_diag['dead'], 'perplexity': _b_diag['perplexity']},
         'confirmed': _confirmed, 'steps': BONUS_STEPS, 'train_time_s': round(BONUS_TIME, 1)}
save_json('bonus.json', bonus)
_lines = ['# Бонус: реактивация dead-кодов VQ-VAE K=64', '',
          f'- Гипотеза (до просмотра результатов): {bonus["hypothesis"]}.',
          f'- base K=64: test MSE {round(_base["test_mse"], 5)}, used {_base["used"]}/64, dead {_base["dead"]}, perplexity {round(_base["perplexity"], 2)}.',
          f'- с реактивацией ({_b_reinits} сбросов каждые {REINIT_EVERY} шагов): test MSE {round(_b_diag["test_mse"], 5)}, used {_b_diag["used"]}/64, dead {_b_diag["dead"]}, perplexity {round(_b_diag["perplexity"], 2)}.',
          '- Вердикт: гипотеза ' + ('ПОДТВЕРЖДЕНА' if _confirmed else 'НЕ подтверждена') +
          ' (сравнение при равном бюджете, общей инициализации и общем потоке батчей; различие - только механизм реактивации).']
(ARTIFACTS / 'bonus.md').write_text('\n'.join(_lines) + '\n', encoding='utf-8')
_total = VQ_RESULTS[16]['train_time_s'] + VQ_RESULTS[64]['train_time_s'] + PRIOR_TIME + CLS_TIME + BONUS_TIME
print('БОНУС:', 'ПОДТВЕРЖДЕНА' if _confirmed else 'НЕ подтверждена',
      f'| dead {_base["dead"]} -> {_b_diag["dead"]} | MSE {round(_base["test_mse"], 5)} -> {round(_b_diag["test_mse"], 5)} '
      f'| perplexity {round(_base["perplexity"], 2)} -> {round(_b_diag["perplexity"], 2)}')
print(f'Время этапов: VQ16 {VQ_RESULTS[16]["train_time_s"]} c + VQ64 {VQ_RESULTS[64]["train_time_s"]} c + '
      f'prior {PRIOR_TIME:.1f} c + классификатор {CLS_TIME:.1f} c + бонус {BONUS_TIME:.1f} c = обучение ~{round(_total, 1)} c '
      '(данные, кодирование и генерации отдельно не хронометрировались)')
print('БОНУС OK: bonus.json, bonus.md')

БОНУС: ПОДТВЕРЖДЕНА | dead 30 -> 0 | MSE 0.01053 -> 0.00846 | perplexity 13.48 -> 38.15
Время этапов: VQ16 26.3 c + VQ64 26.3 c + prior 48.2 c + классификатор 8.7 c + бонус 26.0 c = обучение ~135.5 c (данные, кодирование и генерации отдельно не хронометрировались)
БОНУС OK: bonus.json, bonus.md


## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.

## Итоги (cv16-summary)

**Протокол.** Устройство: cuda. Версии: torch 2.14.1+cu126, torchvision 0.29.1+cu126, numpy 2.4.6. Режим: полный (FAST_DEV_RUN=False). Seed 42; инициализации и потоки батчей детерминированы, чекпоинты с resume.

**Данные.** MNIST train 20000 (2000/класс) / test 2000 (200/класс), диапазон [0,1]; источник torchvision, кэш DATA_ROOT/mnist.

**Задание 1 (архитектура).** Энкодер 28→7×7×64 (две stride-2 свёртки), декодер зеркальный с Sigmoid; параметры: энкодер 70304, декодер 55681, prior 289472, классификатор 20490. Loss: recon MSE + ‖sg(z_e)−e‖² + 0.25·‖z_e−sg(e)‖², квантование ближайшим кодом + straight-through; связность проверена ассертами (градиент доходит до энкодера и кодбука).

**Задание 2 (два кодбука, равный бюджет).** 1000 шагов × батч 256, lr 3e-4, общая инициализация энкодера/декодера и общий поток батчей:

| K | loss first→last | test MSE | used | dead | perplexity | время |
|---|---|---|---|---|---|---|
| 16 | 0.2477 → 0.0709 | 0.01254 | 16/16 | 0 | 9.42 | 26.3 c |
| 64 | 0.2472 → 0.0427 | 0.01053 | 34/64 | 30 | 13.48 | 26.3 c |

**Задание 3 (индексы K=64).** train (20000, 49), test (2000, 49); used 41/64, perplexity токенов 13.54.

**Задания 4–5 (prior).** Причинность проверена ассертом (позиции 0..34 не видят токен 35 и будущее); CE 4.3785 → 0.9150 (perplexity 2.50) за 1000 шагов, время 48.2 c.

**Задание 6 (test CE, без обновления весов).** test CE 0.9416 (perplexity 2.56) | train hold-in 0.8839 | униграмма 2.6057 | равномерное ln(64) 4.1589.

**Задания 7–8 (проверяющий классификатор).** DigitCNN 20490 параметров, независим от VQ-VAE и prior; точность на независимом real test 0.9825 (средняя уверенность 0.9812), время 8.7 c.

**Задания 9–10 (сэмплирование; 8 сэмплов/класс, общий seed схем).** Ориентир близости: средняя L2 real-test до ближайшего другого real-test 8.059.

| схема | agreement | уверенность | разнообразие L2 | L2 до test |
|---|---|---|---|---|
| base T=1.0 | 0.838 | 0.927 | 8.021 | 5.381 |
| lowT T=0.75 | 0.938 | 0.953 | 7.388 | 4.877 |
| topk T=1.0, k=20 | 0.850 | 0.926 | 8.045 | 5.385 |
| topp T=1.0, p=0.90 | 0.925 | 0.958 | 7.595 | 4.964 |

**Бонус (реактивация dead-кодов K=64).** base: dead 30, perplexity 13.48, MSE 0.01053; с реактивацией: dead 0, perplexity 38.15, MSE 0.00846. Вердикт: ПОДТВЕРЖДЕНА.

**Время этапов.** VQ16+VQ64: 52.6 c; prior: 48.2 c; классификатор: 8.7 c; бонус: 26.0 c; всего обучение ≈ 135.5 c (данные, кодирование и генерации отдельно не хронометрировались).